In [1]:
import os, re, glob

ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"

# --- Probe A: where does add_lob_v4 live? ---
print("=== A: add_lob_v4 definition sites ===")
hits = []
for dirpath, _, files in os.walk(os.path.join(ROOT, "src")):
    for f in files:
        if f.endswith(".py"):
            p = os.path.join(dirpath, f)
            try:
                txt = open(p, encoding="utf-8", errors="replace").read()
            except Exception as e:
                print("  !read", p, e); continue
            for m in re.finditer(r"def\s+(add_lob\w*)\s*\(", txt):
                hits.append((os.path.relpath(p, ROOT), m.group(1)))
for rel, name in hits:
    print(f"  {name:16s}  <-  {rel}")
if not hits:
    print("  (no add_lob* defs found under src/)")

# --- Probe B: actual DuckDB files in data/raw ---
print("\n=== B: data/raw contents ===")
raw = os.path.join(ROOT, "data", "raw")
for f in sorted(os.listdir(raw)):
    full = os.path.join(raw, f)
    if os.path.isfile(full):
        print(f"  {f}   ({os.path.getsize(full)//1024} KB)")

# --- Probe B2: keyword -> file resolution the way NB24 globs it ---
print("\n=== B2: keyword -> *.duckdb match ===")
for kw in ["baylor", "methodist", "parkland", "medical", "plano"]:
    matches = [os.path.basename(x) for x in glob.glob(os.path.join(raw, "*.duckdb"))
               if kw in os.path.basename(x).lower()]
    print(f"  {kw:10s} -> {matches}")

# --- Probe C: does the repo carry a Medicare 73721 anchor anywhere? ---
print("\n=== C: any '73721' + medicare/allowed references in src ===")
for dirpath, _, files in os.walk(os.path.join(ROOT, "src")):
    for f in files:
        if f.endswith((".py", ".json", ".csv")):
            p = os.path.join(dirpath, f)
            try:
                txt = open(p, encoding="utf-8", errors="replace").read()
            except Exception:
                continue
            if "73721" in txt and re.search(r"medicare|allowed|mpfs|addendum", txt, re.I):
                print(f"  match in {os.path.relpath(p, ROOT)}")
print("  (done)")

=== A: add_lob_v4 definition sites ===
  add_lob_v2        <-  src\queries.py
  add_lob_v3        <-  src\queries.py
  add_lob_v4        <-  src\queries.py
  add_lob           <-  src\queries.py

=== B: data/raw contents ===
  .gitkeep   (0 KB)
  baylor_university_medical_center-69947_parsed.duckdb   (70156 KB)
  medical_city_alliance_hospital-77912_parsed.duckdb   (262412 KB)
  methodist_dallas_medical_center-6000b_parsed.duckdb   (161804 KB)
  parkland_health-6e88d_parsed.duckdb   (169996 KB)
  texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb   (63500 KB)

=== B2: keyword -> *.duckdb match ===
  baylor     -> ['baylor_university_medical_center-69947_parsed.duckdb']
  methodist  -> ['methodist_dallas_medical_center-6000b_parsed.duckdb']
  parkland   -> ['parkland_health-6e88d_parsed.duckdb']
  medical    -> ['baylor_university_medical_center-69947_parsed.duckdb', 'medical_city_alliance_hospital-77912_parsed.duckdb', 'methodist_dallas_medical_center-6000b_parsed.duckdb']
  

In [3]:
import os, json, re
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"

# locate NB24
nb_path = None
for dp, _, files in os.walk(ROOT):
    if ".git" in dp: continue
    for f in files:
        if f.endswith(".ipynb") and ("five_factor" in f or f.startswith("24")):
            nb_path = os.path.join(dp, f)
            print("FOUND:", os.path.relpath(nb_path, ROOT))
if nb_path:
    doc = json.load(open(nb_path, encoding="utf-8"))
    print(f"\n{len(doc['cells'])} cells total\n")
    for idx, c in enumerate(doc["cells"]):
        if c["cell_type"] != "code": continue
        src = "".join(c["source"])
        if re.search(r"add_lob|dbfile|keyword|670103|medical|glob\(|import", src, re.I):
            print(f"\n----- cell[{idx}] -----\n{src}")
else:
    print("No NB24 found — paste `ls` of your notebooks folder")

FOUND: notebooks\24_five_factor_profit_decomposition_1.ipynb

32 cells total


----- cell[3] -----
# Step 0a — imports + repo path resolver
from __future__ import annotations
import os, sys, json, math, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

def _find_repo_root(markers=("src", "outputs", "queries.py")) -> Path:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand/"src"/"queries.py").exists() or (cand/"queries.py").exists():
            return cand
    return here

REPO = _find_repo_root(); SRC = REPO/"src"; OUT = REPO/"outputs"; RAW = REPO/"data"/"raw"
OUT.mkdir(exist_ok=True, parents=True)
print("REPO:", REPO)
print("SRC :", "OK" if (SRC/"queries.py").exists() else "MISSING (TODO fix path)")
print("RAW :

In [4]:
import os, json, re
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
nb = os.path.join(ROOT, "notebooks", "24_five_factor_profit_decomposition_1.ipynb")
doc = json.load(open(nb, encoding="utf-8"))

want = []
for idx, c in enumerate(doc["cells"]):
    if c["cell_type"] != "code": continue
    src = "".join(c["source"])
    # (a) the import cell for add_lob, (b) the CCN<->dbfile keyword registry
    if re.search(r"import\s+.*add_lob|from\s+\w+\s+import", src) or \
       re.search(r"670103|dbfile|keyword|CCN_|registry|450051", src):
        want.append((idx, src))

print(f"{len(want)} target cell(s)\n")
for idx, src in want:
    print(f"########## cell[{idx}] ##########")
    print(src)
    print("########## end ##########\n")

5 target cell(s)

########## cell[3] ##########
# Step 0a — imports + repo path resolver
from __future__ import annotations
import os, sys, json, math, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

def _find_repo_root(markers=("src", "outputs", "queries.py")) -> Path:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand/"src"/"queries.py").exists() or (cand/"queries.py").exists():
            return cand
    return here

REPO = _find_repo_root(); SRC = REPO/"src"; OUT = REPO/"outputs"; RAW = REPO/"data"/"raw"
OUT.mkdir(exist_ok=True, parents=True)
print("REPO:", REPO)
print("SRC :", "OK" if (SRC/"queries.py").exists() else "MISSING (TODO fix path)")
print("RAW :", "OK" if RAW.exists() else "MISSING (TODO point a

In [5]:
import os, json, re
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
nb = os.path.join(ROOT, "notebooks", "24_five_factor_profit_decomposition_1.ipynb")
doc = json.load(open(nb, encoding="utf-8"))

pat = re.compile(r"add_lob|dbfile|keyword|670103|450051|450021|450015|450771|"
                 r"registry|baylor|methodist|parkland|medical|plano|from\s+\S+\s+import|import\s+\S+", re.I)

for idx, c in enumerate(doc["cells"]):
    if c["cell_type"] != "code": continue
    for ln in "".join(c["source"]).splitlines():
        if pat.search(ln) and not ln.strip().startswith("#"):
            print(f"[{idx:2d}] {ln}")

[ 3] from __future__ import annotations
[ 3] import os, sys, json, math, warnings
[ 3] from pathlib import Path
[ 3] from dataclasses import dataclass
[ 3] import numpy as np
[ 3] import pandas as pd
[ 3] import matplotlib
[ 3] import matplotlib.pyplot as plt
[ 5]     ("450021","Baylor","Baylor University Medical Center","baylor","Dallas","nonprofit","Parkland"),
[ 5]     ("450051","Methodist","Methodist Dallas Medical Center","methodist","Dallas","nonprofit","Parkland"),
[ 5]     ("450015","Parkland","Parkland Health","parkland","Dallas","public","self"),
[ 5]     ("670103","MCA","Medical City (MCA / HCA)","medical","Tarrant","for-profit","JPS"),
[ 5]     ("450771","THR_Plano","Texas Health Presbyterian Plano","plano","Collin","nonprofit","none"),
[ 5] ], columns=["ccn","short","name","dbfile_keyword","county","ownership","safety_net"]).set_index("ccn")
[ 5]     "450021":"mid-high; burden absorbed by Parkland -> markup=negotiation, not subsidy",
[ 5]     "450051":"mid; burden absorbed

In [6]:
import os, json
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
nb = os.path.join(ROOT, "notebooks", "24_five_factor_profit_decomposition_1.ipynb")
doc = json.load(open(nb, encoding="utf-8"))
print("===== FULL cell[11] =====")
print("".join(doc["cells"][11]["source"]))
print("\n===== sys.path lines in cell[3] =====")
for ln in "".join(doc["cells"][3]["source"]).splitlines():
    if "sys.path" in ln or "SRC" in ln:
        print(ln)

===== FULL cell[11] =====
# Step 2a — live commercial loader (+ synthetic stand-in) 
def _resolve_dbfile(keyword: str) -> Path:
    matches = sorted(RAW.glob(f"*{keyword}*.duckdb")) if RAW.exists() else []
    if not matches:
        raise NotImplementedError(f"No DuckDB matched '{keyword}' under {RAW}. Fix RAW / naming map.")
    return matches[0]

def _load_real_commercial(code: str = TARGET_CODE) -> pd.DataFrame:
    """Pull rows from the five DuckDBs, classify LOB, apply NB23 rules; return long frame."""
    if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
    try:
        from queries import query_procedure_rates_agg      # noqa
        from lob import add_lob_v4                          # noqa  TODO confirm module path
    except Exception as e:
        raise NotImplementedError(
            f"Wire live imports before USE_SYNTHETIC=False: {e!r} "
            "(expected src/queries.py::query_procedure_rates_agg + add_lob_v4).")
    frames=[]
    for ccn,row in HOSPITALS.i

In [7]:
import os, json, shutil, re
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
nb   = os.path.join(ROOT, "notebooks", "24_five_factor_profit_decomposition_1.ipynb")

# 0) backup
bak = nb + ".day33.bak"
shutil.copy(nb, bak)
print("backup ->", os.path.basename(bak))

doc = json.load(open(nb, encoding="utf-8"))

def edit_cell(idx, old, new, label):
    src = "".join(doc["cells"][idx]["source"])
    n = src.count(old)
    assert n == 1, f"[{label}] expected 1 match in cell[{idx}], found {n}"
    src = src.replace(old, new)
    doc["cells"][idx]["source"] = src.splitlines(keepends=True)
    print(f"OK  {label}  (cell[{idx}])")

# --- Edit 1: MCA keyword medical -> alliance (cell 5) ---
edit_cell(5, '"medical","Tarrant"', '"alliance","Tarrant"', "MCA keyword -> alliance")

# --- Edit 2: import path lob -> queries (cell 11) ---
edit_cell(11, 'from lob import add_lob_v4', 'from queries import add_lob_v4', "add_lob_v4 import -> queries")

# --- Edit 3: harden _resolve_dbfile against ambiguous match (cell 11) ---
edit_cell(
    11,
    "    return matches[0]",
    "    if len(matches) > 1:\n"
    "        raise ValueError(f\"Ambiguous keyword '{keyword}' matched {len(matches)} DuckDBs: \"\n"
    "                         f\"{[m.name for m in matches]}. Make the keyword unique.\")\n"
    "    return matches[0]",
    "harden _resolve_dbfile",
)

json.dump(doc, open(nb, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print("\nsaved notebook.\n")

# --- confirm the query_procedure_rates_agg signature so the call is right ---
ql = open(os.path.join(ROOT, "src", "queries.py"), encoding="utf-8", errors="replace").read().splitlines()
print("=== query_procedure_rates_agg signature ===")
for i, l in enumerate(ql):
    if re.match(r"\s*def\s+query_procedure_rates_agg\s*\(", l):
        for j in range(i, min(len(ql), i+6)):
            print(f"  {j+1}| {ql[j]}")
        break

backup -> 24_five_factor_profit_decomposition_1.ipynb.day33.bak
OK  MCA keyword -> alliance  (cell[5])
OK  add_lob_v4 import -> queries  (cell[11])
OK  harden _resolve_dbfile  (cell[11])

saved notebook.

=== query_procedure_rates_agg signature ===
  508| def query_procedure_rates_agg(con, code: str, setting: str = 'outpatient') -> pd.DataFrame:
  509|     """
  510|     Aggregated version of query_procedure_rates.
  511|     Returns one row per (payer_name, plan_name, methodology_normalized, billing_class_normalized).
  512| 
  513|     Aggregation:


In [8]:
import os, json, re
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
# find NB23
nb23 = None
for dp,_,fs in os.walk(ROOT):
    if ".git" in dp: continue
    for f in fs:
        if f.endswith(".ipynb") and "commercial_cleanup" in f:
            nb23 = os.path.join(dp,f)
print("NB23:", os.path.relpath(nb23, ROOT) if nb23 else "NOT FOUND")
if nb23:
    doc = json.load(open(nb23, encoding="utf-8"))
    for idx,c in enumerate(doc["cells"]):
        if c["cell_type"]!="code": continue
        for ln in "".join(c["source"]).splitlines():
            if re.search(r"duckdb|\.connect|query_procedure_rates_agg|read_only", ln) and not ln.strip().startswith("#"):
                print(f"[{idx:2d}] {ln}")

NB23: notebooks\23_commercial_cleanup_investigation_1.ipynb
[ 2] import duckdb
[ 2] query_agg = _imp("query_procedure_rates_agg", ["src.queries", "queries"])
[ 2]     raise ImportError("query_procedure_rates_agg not found — check the module path under src/.")
[ 2] for path in sorted(glob.glob(str(ROOT / "data" / "raw" / "*_parsed.duckdb"))):
[ 2]     stem = Path(path).name.replace("_parsed.duckdb", "")
[ 2]     con = duckdb.connect(path, read_only=True)
[ 5] import duckdb
[ 5] for fn in ("query_procedure_rates_agg", "add_lob_v4", "add_lob", "classify_plan"):
[ 5] mfile = next((DATA/"raw").glob("methodist*parsed.duckdb"), None)
[ 5]     con = duckdb.connect(str(mfile), read_only=True)
[ 6] import duckdb
[ 6] mfile = next(raw.glob("methodist*parsed.duckdb"))
[ 6] con = duckdb.connect(str(mfile), read_only=True)
[ 6] df = _q.query_procedure_rates_agg(con, "73721")
[ 6] print("=== query_procedure_rates_agg('73721') ===")
[ 7] import duckdb, pandas as pd, numpy as np
[ 7]     f = next(raw.g

In [9]:
import os, json
ROOT = r"c:\Users\kedha\Documents\dfw-hospital-pricing"
nb   = os.path.join(ROOT, "notebooks", "24_five_factor_profit_decomposition_1.ipynb")
doc  = json.load(open(nb, encoding="utf-8"))

def edit_cell(idx, old, new, label):
    src = "".join(doc["cells"][idx]["source"])
    n = src.count(old)
    assert n == 1, f"[{label}] expected 1 match in cell[{idx}], found {n}"
    doc["cells"][idx]["source"] = src.replace(old, new).splitlines(keepends=True)
    print(f"OK  {label}")

# Edit 4: make duckdb available in the loader (mirror NB23)
edit_cell(11,
    "from queries import query_procedure_rates_agg",
    "import duckdb\n        from queries import query_procedure_rates_agg",
    "add 'import duckdb' to loader")

# Edit 5: open a read-only connection per hospital, pass con (not Path)
edit_cell(11,
    'df = query_procedure_rates_agg(_resolve_dbfile(row["dbfile_keyword"]), code)',
    'con = duckdb.connect(str(_resolve_dbfile(row["dbfile_keyword"])), read_only=True)\n'
    '        try:\n'
    '            df = query_procedure_rates_agg(con, code)\n'
    '        finally:\n'
    '            con.close()',
    "open con per hospital -> query_procedure_rates_agg(con, code)")

json.dump(doc, open(nb, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print("\nsaved.\n=== loader body now ===")
# reprint the loader loop for a sanity check
src = "".join(doc["cells"][11]["source"])
start = src.index("frames=[]")
print(src[start:start+400])

OK  add 'import duckdb' to loader
OK  open con per hospital -> query_procedure_rates_agg(con, code)

saved.
=== loader body now ===
frames=[]
    for ccn,row in HOSPITALS.iterrows():
        con = duckdb.connect(str(_resolve_dbfile(row["dbfile_keyword"])), read_only=True)
        try:
            df = query_procedure_rates_agg(con, code)
        finally:
            con.close()
        df["ccn"], df["short"] = ccn, row["short"]; frames.append(df)
    raw = pd.concat(frames, ignore_index=True)
    raw = add_lob_v4(raw)
    raw 
